# local-ai-server v0.2.0 — `/readyz` and Hot-Reload Demo

This notebook walks through the two Phase 2 observability features:
the **`/readyz` per-backend health probe** and the **`config/models.yaml`
hot-reload** mechanism.

## What is covered

| Step | Description |
|------|-------------|
| 1 | Setup: mint a key, start the gateway, poll `/healthz` until ready |
| 2 | `/readyz` happy path: 200 envelope with per-backend status |
| 3 | Per-backend payload walk-through: what each status value means |
| 4 | `/readyz` 503 path: envelope shape when all backends are unreachable |
| 5 | Hot-reload: mutate `config/models.yaml`, observe `/v1/models` update |
| 6 | Cleanup: kill the gateway, restore `models.yaml`, delete the demo key |

## Prerequisites

- Run `uv sync` once to install all dependencies.
- `ollama serve` must be running on `:11434` for the `/readyz` 200 path
  (step 2 and step 3). If Ollama is not running, the Ollama backend will
  show `status: "unreachable"` and `/readyz` will return **503**.
- Run all cells top-to-bottom in order. **Do not skip the cleanup cell.**
- Open this notebook from the **project root** (run `jupyter notebook` or
  `jupyter lab` from `local-ai-server/`). The setup cell locates the
  project root relative to `Path.cwd()`.

In [ ]:
# Setup: imports and shared helpers used throughout the notebook.
import json
import re
import sqlite3
import subprocess
import time
from pathlib import Path

import httpx
import yaml

# Resolve the project root regardless of where Jupyter was launched from.
# Notebooks in posts/ are one level below the repo root.
_nb_dir = Path.cwd()
if (_nb_dir / "scripts" / "generate_api_key.py").exists():
    PROJECT_ROOT = _nb_dir
elif (_nb_dir.parent / "scripts" / "generate_api_key.py").exists():
    PROJECT_ROOT = _nb_dir.parent
else:
    raise RuntimeError(f"Cannot locate project root from {_nb_dir}")

GATEWAY_URL = "http://127.0.0.1:8000"
GATEWAY_LOG = Path("/tmp/lai-readyz-demo.log")
MODELS_YAML = PROJECT_ROOT / "config" / "models.yaml"
DB_PATH = PROJECT_ROOT / "data" / "keys.db"
DEMO_KEY_NAME = "readyz-demo"

print(f"Project root : {PROJECT_ROOT}")
print(f"Models YAML  : {MODELS_YAML}")
print(f"Gateway URL  : {GATEWAY_URL}")
print(f"Gateway log  : {GATEWAY_LOG}")

## Step 1 — Mint a key and start the gateway

We mint a demo key via `scripts/generate_api_key.py`, then launch
`uvicorn app.main:app` in the background with stdout redirected to
`/tmp/lai-readyz-demo.log`. The hot-reload step (step 5) will grep
that log for the `registry_reloaded` event.

We poll `/healthz` — the always-200 liveness endpoint that requires no
auth — until the gateway is ready. The process handle is stored in
`_gateway_proc`; the cleanup cell terminates it.

In [ ]:
# Mint a demo key (same pattern as the auth notebook).
result = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "scripts/generate_api_key.py",
        "--name",
        DEMO_KEY_NAME,
    ],
    capture_output=True,
    text=True,
    cwd=PROJECT_ROOT,
)
assert result.returncode == 0, (
    f"generate_api_key.py failed:\n{result.stderr}"
)

token: str = result.stdout.strip()
assert re.fullmatch(r"sk-local-[A-Za-z0-9_-]+", token), (
    f"Unexpected token format: {token[:20]}..."
)

prefix: str = token[:12]  # PREFIX_LEN = 12

# Safety: never print the full token in committed notebook output.
print("Key minted successfully.")
print(f"  prefix (12 chars) : {prefix}")
print(f"  token preview     : {prefix}...  (remainder redacted)")

In [ ]:
# Start the gateway and redirect all output to the demo log file.
_log_fh = GATEWAY_LOG.open("w")
_gateway_proc = subprocess.Popen(
    ["uv", "run", "uvicorn", "app.main:app", "--port", "8000"],
    cwd=PROJECT_ROOT,
    stdout=_log_fh,
    stderr=subprocess.STDOUT,
)
print(f"Gateway started (PID {_gateway_proc.pid}). Polling /healthz ...")

deadline = time.time() + 15
ready = False
while time.time() < deadline:
    time.sleep(0.5)
    try:
        resp = httpx.get(f"{GATEWAY_URL}/healthz", timeout=2.0)
        if resp.status_code == 200:
            ready = True
            break
    except httpx.ConnectError:
        pass  # process still starting up

if not ready:
    _gateway_proc.terminate()
    _log_fh.close()
    raise RuntimeError("Gateway did not become ready within 15 s")

print("Gateway ready. /healthz -> 200")

## Step 2 — `/readyz` happy path

`GET /readyz` is a **public endpoint** — no `Authorization` header is
required. It fans out to every backend's `health()` method via
`asyncio.gather` and aggregates the results.

When at least one backend reports `status: "ok"`, the response is
**HTTP 200** with the envelope:

```json
{
  "status": "ok",
  "backends": {
    "ollama": { "status": "ok", "models": ["llama3.1:8b", ...] },
    "mlx":    { "status": "error", "error": "NotSupportedError: ..." },
    "docker_model_runner": { "status": "error", "error": "..." }
  }
}
```

The top-level `status` is binary: `"ok"` (any backend up). There is no
`"degraded"` state in v0.2.0 — that is reserved for a future release.

In [ ]:
# GET /readyz — no Authorization header (public endpoint).
resp = httpx.get(f"{GATEWAY_URL}/readyz", timeout=10.0)

print(f"HTTP status : {resp.status_code}")
body = resp.json()
print(json.dumps(body, indent=2))

In [ ]:
# Assert the envelope shape is correct whether Ollama is up or down.
# (When Ollama is up: 200 + status='ok'.
#  When Ollama is unreachable: 503 + error envelope.)
assert resp.status_code in (200, 503), (
    f"Unexpected status: {resp.status_code}"
)
assert "backends" in body, "'backends' key missing from /readyz response"

if resp.status_code == 200:
    assert body.get("status") == "ok"
    print("Ollama is UP. /readyz -> 200, status='ok'")
else:
    err = body.get("error", {})
    assert err.get("type") == "service_unavailable"
    assert err.get("code") == "no_backends_reachable"
    print("Ollama is DOWN. /readyz -> 503, code='no_backends_reachable'")

print(f"\nBackends in response: {sorted(body['backends'].keys())}")

## Step 3 — Per-backend payload walk-through

Each entry in `backends` has at minimum a `status` field. The possible
values are:

| `status` | When it appears | Extra fields |
|----------|----------------|--------------|
| `"ok"` | Adapter's `health()` returned a dict with `status: "ok"` | `models`: list of upstream model names pulled by Ollama |
| `"unreachable"` | Adapter connected but received no response (e.g., Ollama is stopped) | `error`: a short description |
| `"error"` | Adapter's `health()` raised an exception (e.g., `NotSupportedError` for MLX/DMR stubs) | `error`: `"TypeName: message"` |

In v0.2.0, MLX (`mlx`) and Docker Model Runner (`docker_model_runner`)
are **not yet implemented** — their adapters raise `NotSupportedError`,
which `/readyz` maps to `status: "error"`. As long as Ollama is up,
the aggregated result is still **200** because the rule is:
> 200 if **any** backend is `ok`; 503 only if **none** are.

In [ ]:
# Pretty-print each backend's payload for human reading.
backends = body.get("backends", {})
for name in sorted(backends.keys()):
    payload = backends[name]
    status = payload.get("status", "?")
    extra = ""
    if "models" in payload:
        extra = f"  models={payload['models']}"
    elif "error" in payload:
        # Truncate long error strings for readability.
        err_str = payload["error"]
        extra = f"  error={err_str[:60]}{'...' if len(err_str) > 60 else ''}"
    print(f"  {name:<25} status={status!r}{extra}")

## Step 4 — `/readyz` 503 path (simulated)

We do **not** kill `ollama serve` to demonstrate the 503 path — that
would pollute your local Ollama state. Instead, this cell shows the
**exact 503 envelope** the gateway returns when all backends are
unreachable, and verifies the shape against the architecture spec.

The 503 body is a standard OpenAI error envelope with an extra top-level
`backends` key for ops debugging:

```json
{
  "error": {
    "type": "service_unavailable",
    "message": "No backends reachable",
    "param": null,
    "code": "no_backends_reachable"
  },
  "backends": {
    "docker_model_runner": {
      "status": "error",
      "error": "NotSupportedError: ..."
    },
    "mlx": {
      "status": "error",
      "error": "NotSupportedError: ..."
    },
    "ollama": {
      "status": "unreachable",
      "error": "All connection attempts failed"
    }
  }
}
```

**When does this actually happen?**  
Stop `ollama serve` (or point `OLLAMA_BASE_URL` at a dead port) before
starting the gateway. With Ollama down, `/readyz` returns 503 because
Ollama shows `"unreachable"` and MLX + Docker Model Runner are both
`"error"` (not yet implemented in v0.2.0). None of the three backends
is `"ok"` so the threshold is not met.

In [ ]:
# Validate the 503 envelope shape definition against the architecture
# spec WITHOUT needing to reproduce the failure.

# This is the exact JSON the gateway would return when Ollama is down.
expected_503_shape: dict = {
    "error": {
        "type": "service_unavailable",
        "message": "No backends reachable",
        "param": None,
        "code": "no_backends_reachable",
    },
    "backends": {
        "docker_model_runner": {
            "status": "error",
            "error": (
                "NotSupportedError: "
                "Docker Model Runner adapter is not implemented in v0.1.0"
            ),
        },
        "mlx": {
            "status": "error",
            "error": (
                "NotSupportedError: "
                "MLX adapter is not implemented in v0.1.0"
            ),
        },
        "ollama": {
            "status": "unreachable",
            "error": "All connection attempts failed",
        },
    },
}

# Assert the shape contract: top-level keys, error block fields, and
# backends structure.
err_block = expected_503_shape["error"]
assert err_block["type"] == "service_unavailable"
assert err_block["code"] == "no_backends_reachable"
assert err_block["param"] is None
assert "backends" in expected_503_shape

print("503 envelope shape (as returned when all backends are down):")
print(json.dumps(expected_503_shape, indent=2))

# If /readyz already returned 503 in step 2 (Ollama is not running),
# validate the real response against the expected shape.
if resp.status_code == 503:
    real_err = body.get("error", {})
    assert real_err.get("type") == "service_unavailable", real_err
    assert real_err.get("code") == "no_backends_reachable", real_err
    assert real_err.get("param") is None, real_err
    print("\nLive 503 validated against spec shape.")

## Step 5 — Hot-reload of `config/models.yaml`

The gateway runs a `watchfiles` background task that watches
`config/models.yaml`. On every file save it calls `load_registry()` and
atomically swaps `app.state.registry = new_registry` — one Python
attribute assignment, which CPython's GIL makes atomic relative to
concurrent reads. No process restart is needed.

The steps in this section:

1. Show the current `models.yaml` content.
2. Append a new model entry targeting the existing `ollama` backend.
3. Sleep 1.5 s for the `watchfiles` debounce + reload to complete.
4. `GET /v1/models` (authenticated) — assert the new id appears.
5. Grep the gateway log for the `registry_reloaded` event.
6. **Restore `models.yaml`** to its original content (required — keeps
   the repo clean for the next checkout).

**`/v1/models` requires a Bearer token** — this is the Phase 1 auth
contract. We use the key minted in step 1.

In [ ]:
# Read and display the original models.yaml content.
original_yaml: str = MODELS_YAML.read_text()
print("=== config/models.yaml (original) ===")
print(original_yaml)

In [ ]:
# Append a new model entry and write the file back.
# The watcher picks up the change and reloads within ~1.5 s.
registry_data = yaml.safe_load(original_yaml)

new_entry = {
    "id": "hot-reload-canary",
    "backend": "ollama",
    "upstream_model": "llama3.1:8b",
    "capabilities": ["chat"],
}
registry_data["models"].append(new_entry)

MODELS_YAML.write_text(yaml.safe_dump(registry_data, default_flow_style=False))

print("New entry appended to models.yaml:")
print(json.dumps(new_entry, indent=2))
print("\nWaiting 1.5 s for watchfiles debounce + reload ...")
time.sleep(1.5)
print("Done waiting.")

In [ ]:
# GET /v1/models — requires Bearer token (Phase 1 auth contract).
models_resp = httpx.get(
    f"{GATEWAY_URL}/v1/models",
    headers={"Authorization": f"Bearer {token}"},
    timeout=10.0,
)
assert models_resp.status_code == 200, (
    f"Expected 200, got {models_resp.status_code}: {models_resp.text}"
)

model_ids = [m["id"] for m in models_resp.json()["data"]]
print("Model IDs returned by /v1/models:")
for mid in model_ids:
    marker = "  <-- NEW" if mid == "hot-reload-canary" else ""
    print(f"  - {mid}{marker}")

assert "hot-reload-canary" in model_ids, (
    f"hot-reload-canary not found in {model_ids}"
)
print("\nHot-reload verified: new model id is live without a restart.")

In [ ]:
# Grep the gateway log for the registry_reloaded event emitted by the
# watcher. The log contains one JSON object per line (structlog output).
_log_fh.flush()

reload_events = []
for raw_line in GATEWAY_LOG.read_text().splitlines():
    raw_line = raw_line.strip()
    if not raw_line:
        continue
    try:
        entry = json.loads(raw_line)
    except json.JSONDecodeError:
        continue  # skip non-JSON lines (e.g., uvicorn startup banner)
    if entry.get("event") in (
        "registry_reloaded",
        "registry_reloaded_unchanged",
    ):
        reload_events.append(entry)

if reload_events:
    print(f"Found {len(reload_events)} reload event(s) in the log:")
    for ev in reload_events:
        print(json.dumps(ev, indent=2))
else:
    # The watcher may have not fired yet; print a warning but do not
    # fail — the /v1/models assertion above is the authoritative check.
    print(
        "Warning: no reload event found in log yet."
        " The /v1/models assertion passed, so the reload did happen;"
        " the log may still be buffered."
    )

In [ ]:
# Restore models.yaml to its original content.
# This is required to keep the repo clean for the next checkout.
MODELS_YAML.write_text(original_yaml)
print("models.yaml restored to original content.")

# Brief pause to let the watcher process the restore.
time.sleep(1.5)
print("Restore reload complete.")

## Step 6 — Cleanup

This cell terminates the gateway subprocess, closes the log file handle,
and deletes the demo key row from `data/keys.db`.

`models.yaml` was already restored at the end of step 5. This cell
verifies it is in the clean state before finishing.

In [ ]:
# Terminate the gateway and close the log file.
try:
    _gateway_proc.terminate()
    _gateway_proc.wait(timeout=5)
    print(f"Gateway (PID {_gateway_proc.pid}) terminated.")
except Exception as exc:
    print(f"Warning: could not cleanly terminate gateway: {exc}")
finally:
    _log_fh.close()

# Delete the demo key row from the database.
if DB_PATH.exists():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(
            "DELETE FROM api_keys WHERE prefix = ?", (prefix,)
        )
        remaining = conn.execute(
            "SELECT COUNT(*) FROM api_keys"
        ).fetchone()[0]
    print(f"Demo key deleted. Remaining rows in api_keys: {remaining}")
else:
    print("data/keys.db not found; nothing to clean.")

# Verify models.yaml is clean (no hot-reload-canary entry).
current_yaml = yaml.safe_load(MODELS_YAML.read_text())
ids_now = [m["id"] for m in current_yaml.get("models", [])]
assert "hot-reload-canary" not in ids_now, (
    f"models.yaml still contains hot-reload-canary: {ids_now}"
)
print(f"models.yaml is clean. Model IDs: {ids_now}")
print("Cleanup complete. Re-run from the top to start fresh.")